# MedStock-AU: Pharmaceutical Demand Forecasting
## Module 6 — LLM Replenishment Report Generation + Chatbot API

---

### Overview

This module integrates a Large Language Model (LLM) to:

1. **Auto-generate** human-readable replenishment reports from model outputs
2. **Power a conversational Chatbot API** that pharmacy staff can query in natural language

### Architecture

# MedStock-AU: Pharmaceutical Demand Forecasting

## Module 6 — LLM Replenishment Report Generation + Chatbot API

**Author:** Amanda  
**Date:** April 2026  

---

### Overview

This module integrates a Large Language Model (LLM) to:

1. **Auto-generate** human-readable replenishment reports from model outputs
2. **Power a conversational Chatbot API** that pharmacy staff can query in natural language

### Architecture

## 1. Import Libraries & Load Data

In [2]:
import numpy as np
import pandas as pd
import json
import os
from datetime import datetime, timedelta
from anthropic import Anthropic
import warnings
warnings.filterwarnings('ignore')

# Load all processed data
df = pd.read_csv('../data/processed/pharmacy_demand_with_anomalies.csv', parse_dates=['date'])
forecasts = pd.read_csv('../data/processed/forecasts_rpa_paracetamol.csv', parse_dates=['date'])
model_comparison = pd.read_csv('../data/processed/model_comparison.csv')
rl_results = pd.read_csv('../data/processed/rl_policy_comparison.csv')

print(f"Main dataset shape    : {df.shape}")
print(f"Forecast records      : {len(forecasts)}")
print(f"Latest date in data   : {df['date'].max().date()}")
print(f"\nLocations available:")
for loc in sorted(df['location'].unique()):
    print(f"  - {loc}")
print(f"\nMedications available:")
for med in sorted(df['medication'].unique()):
    print(f"  - {med}")

Main dataset shape    : (104544, 31)
Forecast records      : 92
Latest date in data   : 2024-12-31

Locations available:
  - Chemist Warehouse Epping
  - Chemist Warehouse Sydney CBD
  - Priceline Pharmacy Pitt Street
  - Prince of Wales Hospital Pharmacy
  - RPA Hospital Pharmacy
  - St Vincent's Hospital Pharmacy
  - TerryWhite Chemmart Parramatta
  - Westmead Hospital Pharmacy

Medications available:
  - Amoxicillin
  - Atorvastatin
  - Cetirizine
  - Codeine
  - Dexamethasone
  - Enoxaparin
  - Ibuprofen
  - Metformin
  - Omeprazole
  - Ondansetron
  - Pantoprazole
  - Paracetamol
  - Rosuvastatin
  - Salbutamol
  - Sertraline


## 2. Build Data Query Functions

These functions simulate real-time data retrieval —  
in production, these would query a live database.

In [3]:
def get_current_stock(location, medication):
    """Simulate current stock level based on latest demand data."""
    subset = df[
        (df['location'].str.lower().str.contains(location.lower())) &
        (df['medication'].str.lower().str.contains(medication.lower()))
    ].sort_values('date')
    
    if subset.empty:
        return None
    
    latest = subset.iloc[-1]
    # Simulate stock as reorder_point minus recent demand variance
    avg_demand   = subset.tail(7)['demand_units'].mean()
    reorder_pt   = latest['reorder_point']
    simulated_stock = max(0, int(reorder_pt - avg_demand * np.random.uniform(0.3, 0.9)))
    
    return {
        'location'      : latest['location'],
        'medication'     : latest['medication'],
        'stock_units'    : simulated_stock,
        'reorder_point'  : int(reorder_pt),
        'avg_demand_7d'  : round(avg_demand, 1),
        'unit_cost_aud'  : float(latest['unit_cost_aud']),
        'status'         : 'LOW' if simulated_stock < reorder_pt * 0.5 else
                           'CRITICAL' if simulated_stock < reorder_pt * 0.2 else 'OK',
        'last_updated'   : str(latest['date'].date())
    }


def get_demand_forecast(location, medication, days=7):
    """Return recent demand trend as forecast proxy."""
    subset = df[
        (df['location'].str.lower().str.contains(location.lower())) &
        (df['medication'].str.lower().str.contains(medication.lower()))
    ].sort_values('date').tail(30)
    
    if subset.empty:
        return None
    
    avg_demand  = subset['demand_units'].mean()
    trend       = subset['demand_units'].iloc[-7:].mean() - subset['demand_units'].iloc[:7].mean()
    forecast    = [max(0, int(avg_demand + trend * 0.1 + np.random.normal(0, avg_demand * 0.05)))
                   for _ in range(days)]
    
    return {
        'location'       : subset.iloc[-1]['location'],
        'medication'     : subset.iloc[-1]['medication'],
        'forecast_days'  : days,
        'daily_forecast' : forecast,
        'avg_forecast'   : round(np.mean(forecast), 1),
        'trend'          : 'INCREASING' if trend > 5 else 'DECREASING' if trend < -5 else 'STABLE',
        'anomaly_flag'   : bool(subset.iloc[-1]['if_anomaly'])
    }


def get_anomalies(location=None, medication=None, days=30):
    """Return recent anomaly detections."""
    latest_date = df['date'].max()
    cutoff      = latest_date - timedelta(days=days)
    
    subset = df[df['date'] >= cutoff].copy()
    if location:
        subset = subset[subset['location'].str.lower().str.contains(location.lower())]
    if medication:
        subset = subset[subset['medication'].str.lower().str.contains(medication.lower())]
    
    anomalies = subset[subset['if_anomaly'] == 1][
        ['date', 'location', 'medication', 'demand_units', 'reorder_point']
    ].sort_values('date', ascending=False)
    
    return {
        'total_anomalies' : len(anomalies),
        'period_days'     : days,
        'records'         : anomalies.head(5).to_dict('records')
    }


def get_replenishment_summary():
    """Generate summary stats for all locations."""
    latest_date = df['date'].max()
    recent      = df[df['date'] >= latest_date - timedelta(days=7)]
    
    summary = []
    for loc in df['location'].unique():
        loc_data = recent[recent['location'] == loc]
        summary.append({
            'location'        : loc,
            'location_type'   : loc_data['location_type'].iloc[0],
            'avg_daily_demand': round(loc_data['demand_units'].mean(), 1),
            'anomaly_count'   : int(loc_data['if_anomaly'].sum()),
            'high_demand_meds': loc_data.nlargest(3, 'demand_units')['medication'].tolist()
        })
    
    return summary


print("Data query functions defined.")
print(f"\nTest — RPA Paracetamol stock:")
stock = get_current_stock('RPA', 'Paracetamol')
print(json.dumps(stock, indent=2))    

Data query functions defined.

Test — RPA Paracetamol stock:
{
  "location": "RPA Hospital Pharmacy",
  "medication": "Paracetamol",
  "stock_units": 151,
  "reorder_point": 300,
  "avg_demand_7d": 266.3,
  "unit_cost_aud": 0.05,
  "status": "OK",
  "last_updated": "2024-12-31"
}


## 3. LLM Report Generation

We use Claude API to convert raw data outputs into  
human-readable replenishment reports.

In [4]:
from dotenv import load_dotenv
load_dotenv()

client = Anthropic()
print("Anthropic client initialised successfully.")

Anthropic client initialised successfully.


In [5]:
client = Anthropic()

def generate_replenishment_report(location=None, medication=None):
    """Generate a human-readable replenishment report using Claude."""
    
    # Gather data
    if location and medication:
        stock    = get_current_stock(location, medication)
        forecast = get_demand_forecast(location, medication, days=7)
        anomalies = get_anomalies(location, medication, days=30)
        data_summary = {
            'stock'    : stock,
            'forecast' : forecast,
            'anomalies': anomalies
        }
    else:
        data_summary = {
            'all_locations': get_replenishment_summary(),
            'anomalies'    : get_anomalies(days=7)
        }
    
    prompt = f"""You are a pharmaceutical inventory analyst for Sydney hospital pharmacies.

Based on the following inventory data, generate a concise professional replenishment report.

DATA:
{json.dumps(data_summary, indent=2, default=str)}

Write a report that includes:
1. Current inventory status summary
2. Key risks (stockouts, anomalies)
3. Specific replenishment recommendations with quantities
4. Priority actions (HIGH/MEDIUM/LOW)

Keep it under 300 words. Use clear headings. Be specific with numbers."""

    response = client.messages.create(
        model      = 'claude-sonnet-4-5',
        max_tokens = 600,
        messages   = [{'role': 'user', 'content': prompt}]
    )
    
    return response.content[0].text


# Test report generation
print("Generating replenishment report for RPA — Paracetamol...")
print("=" * 60)
report = generate_replenishment_report('RPA', 'Paracetamol')
print(report)
print("=" * 60)

Generating replenishment report for RPA — Paracetamol...
# PHARMACEUTICAL REPLENISHMENT REPORT
**RPA Hospital Pharmacy | Paracetamol**  
*Report Date: 31 December 2024*

---

## 1. CURRENT INVENTORY STATUS

- **Current Stock:** 210 units
- **Reorder Point:** 300 units
- **Stock Position:** **90 units below reorder threshold** (30% deficit)
- **Recent Demand:** 266.3 units/day (7-day average)
- **Days of Supply:** **0.8 days remaining**

**Status:** CRITICAL - Immediate action required

---

## 2. KEY RISKS

### Imminent Stockout Risk
Current inventory will be depleted within **24 hours** based on average daily demand of 266.6 units/day.

### Demand Anomaly
One anomaly detected on 13 December 2024 (zero demand recorded) suggests potential data recording issue. Otherwise, demand pattern is stable with slight decreasing trend.

---

## 3. REPLENISHMENT RECOMMENDATIONS

### Immediate Order Required
- **Recommended Order Quantity:** 2,100 units (10-day supply)
- **Cost:** $105 AUD
- **Ratio

## 4. Chatbot Core Logic

The chatbot uses Claude to:
1. Understand the user's intent
2. Route to the correct data function
3. Return a natural language response

In [6]:
def chatbot_response(user_message, conversation_history=None):
    """
    Process a user query and return a natural language response.
    Uses Claude to understand intent and route to appropriate data functions.
    """
    if conversation_history is None:
        conversation_history = []

    # System prompt — defines chatbot behaviour and available data
    system_prompt = """You are MedStock Assistant, an AI-powered pharmaceutical inventory chatbot for Sydney hospital and retail pharmacies.

You have access to real-time data for these locations:
- RPA Hospital Pharmacy (Camperdown)
- Westmead Hospital Pharmacy
- St Vincent's Hospital Pharmacy (Darlinghurst)  
- Prince of Wales Hospital Pharmacy (Randwick)
- Chemist Warehouse Epping
- Chemist Warehouse Sydney CBD
- Priceline Pharmacy Pitt Street
- TerryWhite Chemmart Parramatta

Available medications: Paracetamol, Ibuprofen, Amoxicillin, Metformin, Atorvastatin, Omeprazole, Salbutamol, Pantoprazole, Codeine, Ondansetron, Enoxaparin, Dexamethasone, Sertraline, Rosuvastatin, Cetirizine

You can answer questions about:
- Current stock levels and status
- Demand forecasts (next 7 days)
- Anomaly alerts and demand spikes
- Replenishment recommendations
- General inventory insights

Be concise, professional, and specific with numbers. 
If asked about a location or medication not in your system, politely say it's not currently monitored.
Always mention if stock is LOW or CRITICAL."""

    # Build data context based on message content
    data_context = ""
    msg_lower = user_message.lower()
    
    # Extract location and medication from message
    locations_map = {
        'rpa': 'RPA', 'royal prince alfred': 'RPA',
        'westmead': 'Westmead',
        "st vincent": 'St Vincent',
        'prince of wales': 'Prince of Wales',
        'epping': 'Epping',
        'cbd': 'Sydney CBD', 'sydney cbd': 'Sydney CBD',
        'pitt street': 'Pitt Street', 'priceline': 'Pitt Street',
        'parramatta': 'Parramatta', 'terrywhite': 'Parramatta'
    }
    
    meds_map = {
        'paracetamol': 'Paracetamol', 'ibuprofen': 'Ibuprofen',
        'amoxicillin': 'Amoxicillin', 'metformin': 'Metformin',
        'atorvastatin': 'Atorvastatin', 'omeprazole': 'Omeprazole',
        'salbutamol': 'Salbutamol', 'pantoprazole': 'Pantoprazole',
        'codeine': 'Codeine', 'ondansetron': 'Ondansetron',
        'enoxaparin': 'Enoxaparin', 'dexamethasone': 'Dexamethasone',
        'sertraline': 'Sertraline', 'rosuvastatin': 'Rosuvastatin',
        'cetirizine': 'Cetirizine'
    }
    
    detected_loc = next((v for k, v in locations_map.items() if k in msg_lower), None)
    detected_med = next((v for k, v in meds_map.items() if k in msg_lower), None)
    
    # Fetch relevant data
    if detected_loc and detected_med:
        stock    = get_current_stock(detected_loc, detected_med)
        forecast = get_demand_forecast(detected_loc, detected_med)
        anomaly  = get_anomalies(detected_loc, detected_med, days=14)
        data_context = f"\nCURRENT DATA:\n{json.dumps({'stock': stock, 'forecast': forecast, 'anomalies': anomaly}, indent=2, default=str)}"
    elif detected_loc:
        anomaly  = get_anomalies(detected_loc, days=14)
        summary  = [s for s in get_replenishment_summary() if detected_loc.lower() in s['location'].lower()]
        data_context = f"\nCURRENT DATA:\n{json.dumps({'location_summary': summary, 'anomalies': anomaly}, indent=2, default=str)}"
    elif 'anomal' in msg_lower or 'alert' in msg_lower:
        anomaly  = get_anomalies(days=7)
        data_context = f"\nCURRENT DATA:\n{json.dumps({'recent_anomalies': anomaly}, indent=2, default=str)}"
    elif 'report' in msg_lower or 'summary' in msg_lower:
        summary  = get_replenishment_summary()
        data_context = f"\nCURRENT DATA:\n{json.dumps({'all_locations': summary}, indent=2, default=str)}"

    # Add data context to user message
    full_message = user_message + data_context
    
    # Add to conversation history
    conversation_history.append({'role': 'user', 'content': full_message})
    
    # Call Claude
    response = client.messages.create(
        model    = 'claude-sonnet-4-5',
        max_tokens = 400,
        system   = system_prompt,
        messages = conversation_history
    )
    
    reply = response.content[0].text
    conversation_history.append({'role': 'assistant', 'content': reply})
    
    return reply, conversation_history


print("Chatbot function defined.")

Chatbot function defined.


## 5. Test Chatbot Conversations

Simulating real pharmacy staff queries.

In [7]:
def chat(message, history=None):
    """Helper to print formatted chat output."""
    reply, history = chatbot_response(message, history)
    print(f"👤 User: {message}")
    print(f"🤖 MedStock: {reply}")
    print("-" * 60)
    return history

print("=" * 60)
print("MedStock-AU Chatbot — Demo Conversation")
print("=" * 60)

# Test 1: Stock query
history = chat("Does RPA Hospital have Paracetamol in stock?")

# Test 2: Follow-up in same conversation
history = chat("What about the demand forecast for next week?", history)

# Test 3: Different location
history = chat("How about Chemist Warehouse Epping — do they have Ibuprofen?")

# Test 4: Anomaly query
history = chat("Are there any anomalies detected recently at Westmead?")

# Test 5: Full report
history = chat("Generate a replenishment summary for all Sydney locations.")

MedStock-AU Chatbot — Demo Conversation
👤 User: Does RPA Hospital have Paracetamol in stock?
🤖 MedStock: **⚠️ ALERT: LOW STOCK**

Yes, RPA Hospital Pharmacy currently has **70 units** of Paracetamol in stock, but this is **critically low**.

**Key concerns:**
- **Status:** LOW (well below reorder point of 300 units)
- **Average daily demand:** 266 units
- **Stock will last:** Less than 1 day at current demand

**7-day forecast:** Expected demand of 263-298 units daily (avg: 276 units/day)

**Recommendation:** **Urgent replenishment required immediately** to avoid stockout. Suggest ordering at least 2,000+ units to cover the next week and rebuild safety stock.
------------------------------------------------------------
👤 User: What about the demand forecast for next week?
🤖 MedStock: **7-Day Demand Forecast for Paracetamol at RPA Hospital:**

**Daily projected demand:**
- Day 1: 263 units
- Day 2: 287 units
- Day 3: 276 units
- Day 4: 268 units
- Day 5: 262 units
- Day 6: 280 units
- D

## 6. Build FastAPI Backend

In [8]:
# Write FastAPI app to file
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, timedelta
from anthropic import Anthropic

app = FastAPI(
    title       = "MedStock-AU API",
    description = "AI-Powered Pharmaceutical Demand Forecasting and Inventory Optimisation",
    version     = "1.0.0"
)

client = Anthropic()

# Load data on startup
df        = pd.read_csv('../data/processed/pharmacy_demand_with_anomalies.csv', parse_dates=["date"])
forecasts = pd.read_csv('../data/processed/forecasts_rpa_paracetamol.csv', parse_dates=["date"])

# ── Request/Response Models ──────────────────────────────────────
class ChatRequest(BaseModel):
    message: str
    conversation_history: Optional[List[dict]] = []

class ChatResponse(BaseModel):
    reply: str
    conversation_history: List[dict]

class StockRequest(BaseModel):
    location: str
    medication: str

class ForecastRequest(BaseModel):
    location: str
    medication: str
    days: Optional[int] = 7

# ── Helper Functions ─────────────────────────────────────────────
def get_current_stock(location, medication):
    subset = df[
        (df["location"].str.lower().str.contains(location.lower())) &
        (df["medication"].str.lower().str.contains(medication.lower()))
    ].sort_values("date")
    if subset.empty:
        return None
    latest = subset.iloc[-1]
    avg_demand = subset.tail(7)["demand_units"].mean()
    reorder_pt = latest["reorder_point"]
    simulated_stock = max(0, int(reorder_pt - avg_demand * np.random.uniform(0.3, 0.9)))
    return {
        "location"     : latest["location"],
        "medication"   : latest["medication"],
        "stock_units"  : simulated_stock,
        "reorder_point": int(reorder_pt),
        "avg_demand_7d": round(avg_demand, 1),
        "status"       : "LOW" if simulated_stock < reorder_pt * 0.5 else
                         "CRITICAL" if simulated_stock < reorder_pt * 0.2 else "OK",
    }

def get_demand_forecast(location, medication, days=7):
    subset = df[
        (df["location"].str.lower().str.contains(location.lower())) &
        (df["medication"].str.lower().str.contains(medication.lower()))
    ].sort_values("date").tail(30)
    if subset.empty:
        return None
    avg_demand = subset["demand_units"].mean()
    trend = subset["demand_units"].iloc[-7:].mean() - subset["demand_units"].iloc[:7].mean()
    forecast = [max(0, int(avg_demand + trend * 0.1 + np.random.normal(0, avg_demand * 0.05)))
                for _ in range(days)]
    return {
        "location"      : subset.iloc[-1]["location"],
        "medication"    : subset.iloc[-1]["medication"],
        "forecast_days" : days,
        "daily_forecast": forecast,
        "avg_forecast"  : round(np.mean(forecast), 1),
        "trend"         : "INCREASING" if trend > 5 else "DECREASING" if trend < -5 else "STABLE",
    }

# ── API Endpoints ────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "message": "MedStock-AU API is running",
        "version": "1.0.0",
        "endpoints": ["/stock", "/forecast", "/anomalies", "/report", "/chat", "/health"]
    }

@app.get("/health")
def health():
    return {"status": "healthy", "timestamp": str(datetime.now())}

@app.post("/stock")
def stock_endpoint(request: StockRequest):
    result = get_current_stock(request.location, request.medication)
    if not result:
        raise HTTPException(status_code=404, detail="Location or medication not found")
    return result

@app.post("/forecast")
def forecast_endpoint(request: ForecastRequest):
    result = get_demand_forecast(request.location, request.medication, request.days)
    if not result:
        raise HTTPException(status_code=404, detail="Location or medication not found")
    return result

@app.get("/anomalies")
def anomalies_endpoint(location: Optional[str] = None, days: int = 7):
    latest_date = df["date"].max()
    cutoff = latest_date - timedelta(days=days)
    subset = df[df["date"] >= cutoff]
    if location:
        subset = subset[subset["location"].str.lower().str.contains(location.lower())]
    anomalies = subset[subset["if_anomaly"] == 1][
        ["date", "location", "medication", "demand_units"]
    ].sort_values("date", ascending=False)
    return {
        "total_anomalies": len(anomalies),
        "period_days"    : days,
        "records"        : anomalies.head(10).to_dict("records")
    }

@app.get("/report")
def report_endpoint(location: Optional[str] = None):
    latest = df["date"].max()
    recent = df[df["date"] >= latest - timedelta(days=7)]
    if location:
        recent = recent[recent["location"].str.lower().str.contains(location.lower())]
    summary = recent.groupby("location").agg(
        avg_demand   = ("demand_units", "mean"),
        anomaly_count= ("if_anomaly", "sum"),
        total_cost   = ("unit_cost_aud", lambda x: (x * recent.loc[x.index, "demand_units"]).sum())
    ).round(2).reset_index()
    return summary.to_dict("records")

@app.post("/chat")
def chat_endpoint(request: ChatRequest):
    try:
        system_prompt = """You are MedStock Assistant, an AI-powered pharmaceutical inventory chatbot for Sydney pharmacies.
You have data for: RPA Hospital, Westmead Hospital, St Vincent\'s Hospital, Prince of Wales Hospital,
Chemist Warehouse Epping, Chemist Warehouse Sydney CBD, Priceline Pitt Street, TerryWhite Chemmart Parramatta.
Be concise, professional, and specific with numbers."""

        history = request.conversation_history or []
        history.append({"role": "user", "content": request.message})

        response = client.messages.create(
            model     = "claude-sonnet-4-5",
            max_tokens= 400,
            system    = system_prompt,
            messages  = history
        )

        reply = response.content[0].text
        history.append({"role": "assistant", "content": reply})

        return ChatResponse(reply=reply, conversation_history=history)

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

with open('../api/main.py', 'w') as f:
    f.write(api_code)

print("FastAPI backend written to: api/main.py")
print("\nAPI Endpoints:")
print("  GET  /          — API info")
print("  GET  /health    — Health check")
print("  POST /stock     — Current stock query")
print("  POST /forecast  — Demand forecast")
print("  GET  /anomalies — Recent anomalies")
print("  GET  /report    — Location summary report")
print("  POST /chat      — Chatbot conversation")

FastAPI backend written to: api/main.py

API Endpoints:
  GET  /          — API info
  GET  /health    — Health check
  POST /stock     — Current stock query
  POST /forecast  — Demand forecast
  GET  /anomalies — Recent anomalies
  GET  /report    — Location summary report
  POST /chat      — Chatbot conversation


## 7. Summary

In [9]:
print("=" * 60)
print("MedStock-AU — Module 6 Complete")
print("=" * 60)
print("""
Components built:
  LLM replenishment report generation (Claude API)
  Conversational chatbot with intent routing
  Multi-turn conversation support
  FastAPI backend with 6 endpoints
  Natural language interface for pharmacy staff

Chatbot capabilities demonstrated:
  Stock level queries by location + medication
  7-day demand forecasts
  Anomaly alerts
  Full Sydney network replenishment summary

API ready to run with:
  $ cd api && uvicorn main:app --reload --port 8000
""")

MedStock-AU — Module 6 Complete

Components built:
  LLM replenishment report generation (Claude API)
  Conversational chatbot with intent routing
  Multi-turn conversation support
  FastAPI backend with 6 endpoints
  Natural language interface for pharmacy staff

Chatbot capabilities demonstrated:
  Stock level queries by location + medication
  7-day demand forecasts
  Anomaly alerts
  Full Sydney network replenishment summary

API ready to run with:
  $ cd api && uvicorn main:app --reload --port 8000

